# 05 Analysis master

**Owner:** Sahil

1. Join cohort + real outcomes + club strength + nation rank into `data/clean/analysis.parquet` (one row per cohort entrant).
2. **Calibration (the story's ending):** what share of each potential band reached its potential in the game, and
   what share "made it" in real life.
3. Compute every headline number the chart titles need and write it to `outputs/findings.json`.
4. Propose candidates for the **featured five** (FIFA 15 entrants where possible). Nothing is finalised here.

Conventions (CLAUDE.md §6): gap = potential at entry − peak later rating (positive = overrated); vanished players have no
gap and stay their own category; already-peaked players are excluded from accuracy metrics. Real-life "made it" =
≥ 3,000 Big-5 league minutes after entry.

**Provisional until the manual reviews are done:** Transfermarkt matches still pending in `reports/match_review.csv`,
and the seeded club-name overrides. Re-run this notebook after the reviews.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src import config, dq, findings as fd, outcomes

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)

CHECKS, NOTES = [], []
def check(name, passed, detail=""):
    CHECKS.append({"Check": name, "Result": "PASS" if passed else "FAIL", "Detail": detail})
    print(("PASS" if passed else "FAIL"), "-", name, ("| " + detail if detail else ""))
    return passed

NB = "05_analysis_master"

def spearman(x, y):
    # Spearman rank correlation on pairwise-complete rows (Pearson on ranks; no scipy needed)
    m = x.notna() & y.notna()
    return x[m].rank().corr(y[m].rank())
R, FS, V = config.STATUS_REACHED, config.STATUS_FELL_SHORT, config.STATUS_VANISHED

## 1. Join into `analysis.parquet`

In [2]:
coh = pd.read_parquet(config.CLEAN / "cohort.parquet")
xw = pd.read_parquet(config.CLEAN / "tm_crosswalk.parquet")
real = pd.read_parquet(config.CLEAN / "real_outcomes.parquet").drop(columns=["entry_date"])
cs = pd.read_parquet(config.CLEAN / "club_strength.parquet")
nr = pd.read_parquet(config.CLEAN / "nation_rank.parquet")
ni = pd.read_parquet(config.CLEAN / "nation_iso.parquet")

a = coh.merge(xw[["fifa_player_id", "tm_player_id", "match_method", "needs_review"]].rename(columns={"fifa_player_id": "player_id"}),
              on="player_id", how="left", validate="one_to_one")
a = a.merge(real.drop(columns=["tm_player_id"]).rename(columns={"fifa_player_id": "player_id"}), on="player_id", how="left",
            validate="one_to_one")
a = a.merge(cs[["player_id", "clubelo_club", "club_elo_at_entry", "club_strength_status", "non_european_club"]],
            on="player_id", how="left", validate="one_to_one")
a = a.merge(nr[["fifa_nation", "edition", "nation_rank"]].rename(columns={"fifa_nation": "nationality_name", "edition": "entry_edition",
                                                                           "nation_rank": "nation_rank_at_entry"}),
            on=["nationality_name", "entry_edition"], how="left", validate="many_to_one")
a = a.merge(ni[["fifa_nation", "iso3", "confederation"]].rename(columns={"fifa_nation": "nationality_name"}),
            on="nationality_name", how="left", validate="many_to_one")

a["is_matched"] = a["tm_player_id"].notna()
# Real-life "made it": unmatched entrants count as not made it. Every player with a Big-5 appearance is in
# Transfermarkt's players table (notebook 02), so an unmatched entrant has no Big-5 minutes unless his match was missed.
for c_ in [f"made_it_{t}" for t in config.MADE_IT_MINUTES_SENSITIVITY] + [f"made_it_{config.FOLLOWUP_YEARS}y", "made_it_top_flight"]:
    a[c_] = a[c_].fillna(False).astype(bool)
a["made_it"] = a[f"made_it_{config.MADE_IT_MINUTES_DEFAULT}"]
a["big5_minutes"] = a["big5_minutes"].fillna(0).where(a["is_matched"])
a["in_accuracy_sample"] = ~a["already_peaked"]

comps = pd.read_csv(config.RAW_TM / "competitions.csv")
current_season = int(pd.read_csv(config.RAW_TM / "players.csv", usecols=["last_season"])["last_season"].max())
a["latest_club_tier"] = outcomes.latest_club_tier(a, comps, current_season)

check("analysis: one row per cohort entrant", len(a) == len(coh) and a["player_id"].is_unique, f"{len(a):,}")
check("Status counts sum to the cohort size", int(a["status"].value_counts().sum()) == len(a))
a.to_parquet(config.CLEAN / "analysis.parquet", index=False)
print("analysis.parquet:", a.shape)

n_peaked = int(a["already_peaked"].sum())
print(f"Already peaked at entry (excluded from accuracy metrics): {n_peaked}")
acc = a[a["in_accuracy_sample"]]

PASS - analysis: one row per cohort entrant | 11,519
PASS - Status counts sum to the cohort size 
analysis.parquet: (11519, 143)
Already peaked at entry (excluded from accuracy metrics): 0


## 2. Calibration: the game vs real life
Shares are of all entrants in the band (vanished players are in the denominator as their own category). Mean gap is over
players with a later edition.

In [3]:
def band_table(df):
    g = df.groupby("potential_band", observed=True)
    t = pd.DataFrame({
        "n": g.size(),
        "reached_%": g["status"].apply(lambda s: (s == R).mean() * 100),
        "fell_short_%": g["status"].apply(lambda s: (s == FS).mean() * 100),
        "vanished_%": g["status"].apply(lambda s: (s == V).mean() * 100),
        "mean_gap": g["gap"].apply(lambda s: s.astype(float).mean()),
        "matched_to_TM_%": g["is_matched"].mean() * 100,
        "made_it_real_%": g["made_it"].mean() * 100,
        "made_it_real_matched_only_%": g.apply(lambda d: d.loc[d["is_matched"], "made_it"].mean() * 100, include_groups=False),
        "made_it_1500_%": g["made_it_1500"].mean() * 100,
        "made_it_5000_%": g["made_it_5000"].mean() * 100,
    })
    t.index = t.index.astype(str)
    return t

cal = band_table(acc)
tot = band_table(acc.assign(potential_band="All"))
cal = pd.concat([cal, tot])
print("=" * 100)
print("CALIBRATION: did the game's potential predict careers?  (cohort entrants FIFA 15–18, aged 21 or under)")
print("=" * 100)
display(cal.round(1))

CALIBRATION: did the game's potential predict careers?  (cohort entrants FIFA 15–18, aged 21 or under)


,n,reached_%,fell_short_%,vanished_%,mean_gap,matched_to_TM_%,made_it_real_%,made_it_real_matched_only_%,made_it_1500_%,made_it_5000_%
potential_band,,,,,,,,,,
<70,5840,29.1,37.3,33.6,1.2,40.7,3.9,9.6,5.1,3.0
70–74,3191,19.7,60.6,19.7,4.5,65.8,9.7,14.7,12.2,7.6
75–79,1784,16.3,75.3,8.5,6.1,81.8,22.4,27.4,26.8,18.0
80–84,600,16.8,81.2,2.0,6.4,90.3,45.8,50.7,52.8,40.0
85+,104,10.6,85.6,3.8,6.8,98.1,68.3,69.6,73.1,61.5
All,11519,23.7,52.3,23.9,3.5,57.1,11.1,19.5,13.5,9.1


In [4]:
# The same question within each band: did 'reached in the game' line up with 'made it' in real life?
xt = (acc[acc["potential"] >= 75].groupby(["potential_band", "status"], observed=True)["made_it"]
         .agg(n="size", made_it_pct=lambda s: round(s.mean() * 100, 1)).unstack("status"))
display(xt)
by_ed = (acc.groupby("entry_edition_label")
            .agg(n=("player_id", "size"), wonderkids=("is_wonderkid", "sum"),
                 reached_pct=("status", lambda s: round((s == R).mean() * 100, 1)),
                 vanished_pct=("status", lambda s: round((s == V).mean() * 100, 1)),
                 made_it_pct=("made_it", lambda s: round(s.mean() * 100, 1))))
display(by_ed)

n                  made_it_pct                 
status         Fell short Reached Vanished  Fell short Reached Vanished
potential_band                                                         
75–79                1343     290      151        13.1    76.9      0.7
80–84                 487     101       12        36.6    96.0      0.0
85+                    89      11        4        67.4   100.0      0.0

,n,wonderkids,reached_pct,vanished_pct,made_it_pct
entry_edition_label,,,,,
FIFA 15,4594,48,34.0,20.2,14.1
FIFA 16,2521,13,22.0,26.8,9.0
FIFA 17,2250,21,15.3,26.4,9.6
FIFA 18,2154,22,12.4,26.0,8.9


## 3. Headline numbers by chart

In [5]:
F = {}
f15 = acc[acc["entry_edition"] == 15]
wk15 = f15[f15["is_wonderkid"]]
F["cohort"] = {
    "entrants": len(a), "entrants_by_edition": a["entry_edition_label"].value_counts().sort_index().to_dict(),
    "already_peaked_excluded": n_peaked, "wonderkids": int(a["is_wonderkid"].sum()),
    "status_counts": a["status"].value_counts().to_dict(), "vanished": int((a["status"] == V).sum()),
    "vanished_pct": fd.pct((a["status"] == V).mean()),
    "reached_pct": fd.pct((acc["status"] == R).mean()), "mean_gap": fd.gap1(acc["gap"].astype(float).mean()),
    "made_it_pct": fd.pct(acc["made_it"].mean()),
}
F["match"] = {
    "matched": int(a["is_matched"].sum()), "match_rate_pct": fd.pct(a["is_matched"].mean()),
    "match_rate_pct_by_band": {str(k): fd.pct(v) for k, v in a.groupby("potential_band", observed=True)["is_matched"].mean().items()},
    "pending_review_players": int(a["needs_review"].fillna(False).sum()),
}
F["calibration_by_band"] = {b: {"n": int(r["n"]), "reached_pct": fd.pct(r["reached_%"] / 100),
                                 "fell_short_pct": fd.pct(r["fell_short_%"] / 100), "vanished_pct": fd.pct(r["vanished_%"] / 100),
                                 "mean_gap": fd.gap1(r["mean_gap"]), "made_it_pct": fd.pct(r["made_it_real_%"] / 100),
                                 "made_it_matched_only_pct": fd.pct(r["made_it_real_matched_only_%"] / 100),
                                 "made_it_1500_pct": fd.pct(r["made_it_1500_%"] / 100), "made_it_5000_pct": fd.pct(r["made_it_5000_%"] / 100)}
                             for b, r in cal.iterrows()}

# Chart 1: FIFA 15 wonderkids
F["chart_01"] = {"fifa15_wonderkids": len(wk15), "reached": int((wk15["status"] == R).sum()),
                 "fell_short": int((wk15["status"] == FS).sum()), "vanished": int((wk15["status"] == V).sum()),
                 "mean_gap": fd.gap1(wk15["gap"].astype(float).mean()),
                 "made_it": int(wk15["made_it"].sum()), "made_it_pct": fd.pct(wk15["made_it"].mean())}

# Chart 2: under-21s with potential >= 85 in each edition (every edition, not first entry)
panel = pd.read_parquet(config.CLEAN / "fifa_panel.parquet", columns=["edition", "edition_label", "age", "potential"])
wk_ed = (panel[(panel["age"] <= config.MAX_ENTRY_AGE) & (panel["potential"] >= config.WONDERKID_POTENTIAL)]
         .groupby(["edition", "edition_label"]).size().reset_index(name="wonderkids"))
check("Chart 2 counts match CLAUDE.md §7", wk_ed.set_index("edition")["wonderkids"].to_dict() == config.EXPECTED_U21_POT85_PER_EDITION)
F["chart_02"] = {"by_edition": dict(zip(wk_ed["edition_label"], wk_ed["wonderkids"])),
                 "first": int(wk_ed["wonderkids"].iat[0]), "last": int(wk_ed["wonderkids"].iat[-1]),
                 "max": int(wk_ed["wonderkids"].max()), "max_edition": wk_ed.loc[wk_ed["wonderkids"].idxmax(), "edition_label"],
                 "ratio_max_to_first": round(wk_ed["wonderkids"].max() / wk_ed["wonderkids"].iat[0], 1)}

# Chart 4: potential >= 75 scatter
p75 = acc[(acc["potential"] >= 75) & acc["peak_later"].notna()]
F["chart_04"] = {"n_points": int((acc["potential"] >= 75).sum()), "n_with_later_edition": len(p75),
                 "below_diagonal_pct": fd.pct((p75["peak_later"] < p75["potential"]).mean())}

# Chart 5: calibration ladder (same as calibration_by_band)
F["chart_05"] = {b: F["calibration_by_band"][b] for b in config.BAND_ORDER}
display(pd.DataFrame(F["chart_05"]).T)

PASS - Chart 2 counts match CLAUDE.md §7 


,n,reached_pct,fell_short_pct,vanished_pct,mean_gap,made_it_pct,made_it_matched_only_pct,made_it_1500_pct,made_it_5000_pct
<70,5840.0,29.0,37.0,34.0,1.2,4.0,10.0,5.0,3.0
70–74,3191.0,20.0,61.0,20.0,4.5,10.0,15.0,12.0,8.0
75–79,1784.0,16.0,75.0,8.0,6.1,22.0,27.0,27.0,18.0
80–84,600.0,17.0,81.0,2.0,6.4,46.0,51.0,53.0,40.0
85+,104.0,11.0,86.0,4.0,6.8,68.0,70.0,73.0,62.0


### Chart 6: busts and bloomers
**Busts:** largest gap among **wonderkids** (potential ≥ 85), where the promise was biggest. Ranked by raw gap, the top busts
are obscure lower-league players. **Bloomers:** most negative gap among entrants whose peak later rating reached 80+.
Ties: larger potential first (busts), higher peak first (bloomers), then name.

In [6]:
cols = ["player_id", "short_name", "long_name", "entry_edition_label", "club_name", "nationality_name", "position_group",
        "potential", "peak_later", "gap", "big5_minutes", "made_it"]
busts = (acc[acc["is_wonderkid"] & acc["peak_later"].notna()]
           .sort_values(["gap", "potential", "short_name"], ascending=[False, False, True]).head(10)[cols])
bloomers = (acc[acc["peak_later"] >= 80]
              .sort_values(["gap", "peak_later", "short_name"], ascending=[True, False, True]).head(10)[cols])
print("Top 10 busts"); display(busts)
print("Top 10 bloomers"); display(bloomers)
def plist(df):
    return [{"player_id": int(r.player_id), "player": r.short_name, "entry_edition": r.entry_edition_label,
             "potential": int(r.potential), "peak_later": int(r.peak_later), "gap": int(r.gap)} for r in df.itertuples()]
F["chart_06"] = {"busts": plist(busts), "bloomers": plist(bloomers),
                 "biggest_bust": plist(busts.head(1))[0], "biggest_bloomer": plist(bloomers.head(1))[0]}

Top 10 busts


,player_id,short_name,long_name,entry_edition_label,club_name,nationality_name,position_group,potential,peak_later,gap,big5_minutes,made_it
10968,240000,E. Abouchabaka,Elias Abouchabaka,FIFA 18,RB Leipzig,Germany,MID,85,63,22,0.0,False
1988,212455,H. Mastour,Hachim Mastour,FIFA 16,Málaga,Morocco,MID,87,68,19,5.0,False
7942,231395,José Gomes,José Gomes,FIFA 17,Benfica,Portugal,FWD,85,66,19,NaN,False
8901,234317,V. Thill,Vincent Thill,FIFA 18,Metz,Luxembourg,MID,85,66,19,26.0,False
9270,235266,C. Früchtl,Christian Früchtl,FIFA 18,FC Bayern München,Germany,GK,86,68,18,9.0,False
10646,239249,Oriol Busquets,Oriol Busquets Mas,FIFA 18,FC Barcelona B,Spain,MID,85,67,18,196.0,False
9699,236248,B. Woodburn,Benjamin Luke Woodburn,FIFA 18,Liverpool,Wales,FWD,85,68,17,6.0,False
10361,238232,Han Kwang Song,Kwang Song Han,FIFA 18,Perugia,Korea DPR,FWD,86,70,16,276.0,False
1895,212210,R. Živković,Richairo Živković,FIFA 15,Ajax,Netherlands,FWD,86,70,16,106.0,False
10954,239981,J. Arp,Jann-Fiete Arp,FIFA 18,Hamburger SV,Germany,FWD,85,69,16,1544.0,False


Top 10 bloomers


,player_id,short_name,long_name,entry_edition_label,club_name,nationality_name,position_group,potential,peak_later,gap,big5_minutes,made_it
4906,224179,Borja Iglesias,Borja Iglesias Quintás,FIFA 15,Celta de Vigo,Spain,FWD,55,83,-28,14900.0,True
5542,225375,K. Laimer,Konrad Laimer,FIFA 15,Salzburg,Austria,MID,57,83,-26,13592.0,True
4738,223848,S. Milinković-Savić,Sergej Milinković-Savić,FIFA 15,Genk,Serbia,MID,65,86,-21,21564.0,True
5632,225508,E. Bailly,Eric Bertrand Bailly,FIFA 15,Espanyol,Côte d'Ivoire,DEF,63,84,-21,11441.0,True
11014,240091,G. Vicario,Guglielmo Vicario,FIFA 18,Venezia,Italy,GK,61,82,-21,14934.0,True
3845,221479,D. Calvert-Lewin,Dominic Calvert-Lewin,FIFA 15,Sheffield United,England,FWD,60,81,-21,18492.0,True
3931,221697,O. Watkins,Ollie Watkins,FIFA 15,Exeter City,England,FWD,59,80,-21,18110.0,True
3726,221087,Pau López,Pau López Sabata,FIFA 15,Espanyol,Spain,GK,63,83,-20,22896.0,True
4461,223113,K. Piątek,Krzysztof Piątek,FIFA 16,Zagłębie Lubin,Poland,FWD,60,80,-20,9928.0,True
5091,224458,Diogo Jota,Diogo José Teixeira da Silva,FIFA 15,Paços de Ferreira,Portugal,MID,66,85,-19,11632.0,True


### Chart 7: did EA learn? (learning cohorts, peak within 4 editions)

In [7]:
lc = pd.read_parquet(config.CLEAN / "learning_cohorts.parquet")
lca = lc[~lc["already_peaked"]]
learn = (lca.groupby("entry_edition_label")
            .agg(n=("player_id", "size"),
                 reached_within_4=("status_within_4", lambda s: (s == R).mean()),
                 mean_gap_within_4=("gap_within_4", lambda s: s.astype(float).mean())))
display(learn.round(3))
F["chart_07"] = {"by_entry_edition": {k: {"n": int(r.n), "reached_within_4_pct": fd.pct(r.reached_within_4),
                                          "mean_gap_within_4": fd.gap1(r.mean_gap_within_4)} for k, r in learn.iterrows()},
                 "first_reached_pct": fd.pct(learn["reached_within_4"].iat[0]), "last_reached_pct": fd.pct(learn["reached_within_4"].iat[-1]),
                 "first_mean_gap": fd.gap1(learn["mean_gap_within_4"].iat[0]), "last_mean_gap": fd.gap1(learn["mean_gap_within_4"].iat[-1]),
                 "already_peaked_excluded": int(lc["already_peaked"].sum())}

,n,reached_within_4,mean_gap_within_4
entry_edition_label,,,
FIFA 15,4594,0.250,2.762
FIFA 16,2521,0.152,4.972
FIFA 17,2250,0.098,6.792
FIFA 18,2154,0.083,7.559
FIFA 19,2207,0.076,8.054
FIFA 20,2187,0.059,8.763


### Charts 8–11: who the game misjudged (nation, club strength, position)

In [8]:
g = acc[acc["peak_later"].notna()].copy()
g["gap_f"] = g["gap"].astype(float)

nat = (g.groupby("nationality_name").agg(n=("player_id", "size"), mean_gap=("gap_f", "mean"),
                                         mean_rank=("nation_rank_at_entry", "mean"),
                                         reached=("status", lambda s: (s == R).mean())))
nat = nat[nat["n"] >= config.CHART_MIN_NATION_N].sort_values("mean_gap")
rho_nat = spearman(nat["mean_rank"], nat["mean_gap"])
F["chart_08"] = {"nations": len(nat), "min_n": config.CHART_MIN_NATION_N,
                 "least_overrated": {"nation": nat.index[0], "mean_gap": fd.gap1(nat["mean_gap"].iat[0]), "n": int(nat["n"].iat[0])},
                 "nations_with_positive_mean_gap": int((nat["mean_gap"] > 0).sum()),
                 "most_overrated": {"nation": nat.index[-1], "mean_gap": fd.gap1(nat["mean_gap"].iat[-1]), "n": int(nat["n"].iat[-1])}}
F["chart_09"] = {"nations": len(nat), "spearman_rank_vs_gap": round(rho_nat, 2)}
print(f"Nations n >= {config.CHART_MIN_NATION_N}: {len(nat)}; Spearman(mean nation rank, mean gap) = {rho_nat:.2f}")
display(pd.concat([nat.head(5), nat.tail(5)]).round(2))

clb = (g[g["club_elo_at_entry"].notna()].groupby("club_name")
         .agg(n=("player_id", "size"), mean_gap=("gap_f", "mean"), mean_elo=("club_elo_at_entry", "mean")))
clb = clb[clb["n"] >= config.CHART_MIN_CLUB_N]
rho_club = spearman(clb["mean_elo"], clb["mean_gap"])
rho_club_players = spearman(g["club_elo_at_entry"], g["gap_f"])
F["chart_10"] = {"clubs": len(clb), "min_n": config.CHART_MIN_CLUB_N, "spearman_elo_vs_gap_clubs": round(rho_club, 2),
                 "spearman_elo_vs_gap_players": round(rho_club_players, 2)}
print(f"Clubs n >= {config.CHART_MIN_CLUB_N} with Elo: {len(clb)}; Spearman(club Elo, mean gap) = {rho_club:.2f} "
      f"(player level {rho_club_players:.2f})")

pos = (g.groupby("primary_position").agg(n=("player_id", "size"), mean_potential=("potential", "mean"),
                                         mean_peak=("peak_later", lambda s: s.astype(float).mean()), mean_gap=("gap_f", "mean"))
         .sort_values("mean_gap"))
display(pos.round(2))
F["chart_11"] = {"positions": {p: {"n": int(r.n), "mean_potential": int(round(r.mean_potential)),
                                   "mean_peak_later": int(round(r.mean_peak)), "mean_gap": fd.gap1(r.mean_gap)} for p, r in pos.iterrows()},
                 "most_overrated_position": pos.index[-1], "least_overrated_position": pos.index[0]}

Nations n >= 15: 56; Spearman(mean nation rank, mean gap) = 0.04


,n,mean_gap,mean_rank,reached
nationality_name,,,,
Mali,21,0.67,60.19,0.52
Bosnia and Herzegovina,23,0.83,23.87,0.52
Denmark,196,0.98,33.4,0.48
Albania,17,1.12,53.41,0.47
Côte d'Ivoire,50,1.14,33.32,0.42
Russia,118,5.79,30.44,0.19
Georgia,17,5.88,115.18,0.12
South Africa,20,5.90,66.1,0.15
Turkey,143,6.31,32.47,0.17


Clubs n >= 5 with Elo: 397; Spearman(club Elo, mean gap) = 0.04 (player level 0.02)


,n,mean_potential,mean_peak,mean_gap
primary_position,,,,
RB,586,69.46,66.93,2.53
CDM,497,70.81,68.23,2.58
LB,622,69.81,67.17,2.64
CB,1279,70.29,67.34,2.95
CM,1229,69.72,66.43,3.29
LM,452,71.00,67.67,3.33
RM,437,71.23,67.72,3.51
GK,917,68.28,64.59,3.69
LW,237,74.32,70.22,4.10


### Chart 12: which teenage attributes mattered (outfield players)

In [9]:
skip = {"goalkeeping_diving", "goalkeeping_handling", "goalkeeping_kicking", "goalkeeping_positioning",
        "goalkeeping_reflexes", "goalkeeping_speed"}
attr_cols = config.FACE_STATS + [c for c in coh.columns if c.split("_")[0] in
            {"attacking", "skill", "movement", "power", "mentality", "defending"} and c not in skip]
attr_cols = list(dict.fromkeys(attr_cols + ["weak_foot", "skill_moves"]))   # in-game rated attributes only, no duplicates
other_cols = ["age", "height_cm", "weight_kg"]                                 # not FIFA attributes: reported separately
o = g[~g["is_gk"]].copy()
o["beat_potential"] = -o["gap_f"]   # peak later - potential
corr = o[attr_cols].apply(lambda s: spearman(s, o["beat_potential"])).sort_values()
other = o[other_cols].apply(lambda s: spearman(s, o["beat_potential"]))
check("Chart 12: attribute list has no duplicates", len(attr_cols) == len(set(attr_cols)), f"{len(attr_cols)} attributes")
display(corr.round(3).to_frame("spearman"))
print("Not attributes (reported separately):"); display(other.round(3).to_frame("spearman"))
F["chart_12"] = {"n_outfield": len(o), "n_attributes": len(attr_cols),
                 "top_positive": {k: round(v, 2) for k, v in corr.tail(8)[::-1].items()},
                 "top_negative": {k: round(v, 2) for k, v in corr.head(8).items()},
                 "max_abs_spearman": round(float(corr.abs().max()), 2),
                 "non_attribute_spearman": {k: round(v, 2) for k, v in other.items()}}

PASS - Chart 12: attribute list has no duplicates | 37 attributes


,spearman
skill_moves,-0.100
attacking_finishing,-0.096
shooting,-0.091
mentality_penalties,-0.090
skill_dribbling,-0.085
dribbling,-0.082
attacking_volleys,-0.081
mentality_positioning,-0.072
movement_balance,-0.070
skill_curve,-0.069


Not attributes (reported separately):


,spearman
age,0.202
height_cm,0.073
weight_kg,0.072


### Chart 14 (provisional tier rule, notebook 12 owns the chart)

In [10]:
tiers = acc[acc["potential"] >= 75].groupby(["potential_band", "status", "latest_club_tier"], observed=True).size()
tier_share = (acc[acc["potential"] >= 75]["latest_club_tier"].value_counts(normalize=True)
                .reindex(outcomes.TIER_ORDER).fillna(0))
display(tier_share.round(3).to_frame("share of potential-75+"))
wk_tier = acc[acc["is_wonderkid"]]["latest_club_tier"].value_counts().reindex(outcomes.TIER_ORDER).fillna(0).astype(int)
F["chart_14"] = {"potential_75plus_tier_pct": {k: fd.pct(v) for k, v in tier_share.items()},
                 "wonderkids_by_tier": wk_tier.to_dict(), "current_season": current_season, "provisional": True}

,share of potential-75+
latest_club_tier,
Big-5,0.172
Other European top flight,0.146
Other or lower league,0.406
Outside Europe,0.097
No current club,0.025
Not in Transfermarkt,0.154


## 4. Deeper tests (95% bootstrap confidence intervals)
These test the stronger story: **potential ranks talent about as well as simple benchmarks and the transfer market, but it is
overconfident, most of all for the youngest players and for prospects at weaker clubs, and EA handed out more promise over time.**
Every interval is a 95% percentile bootstrap over players (resample count `BOOTSTRAP_REPS` and seed `RANDOM_SEED` in
`src/config.py`). A finding counts only if its interval excludes the "no effect" value.

In [11]:
rng = np.random.default_rng(config.RANDOM_SEED)
B = config.BOOTSTRAP_REPS

def boot_ci(df, stat, B=B):
    # 95% percentile bootstrap CI of stat(df) over rows
    n = len(df)
    vals = np.array([stat(df.iloc[rng.integers(0, n, n)]) for _ in range(B)], dtype=float)
    lo, hi = np.nanpercentile(vals, [2.5, 97.5])
    return float(stat(df)), float(lo), float(hi)

def auc(score, y):
    # Area under the ROC curve via ranks (Mann-Whitney); 0.5 = coin flip
    m = score.notna() & y.notna()
    s, t = score[m], y[m].astype(bool)
    n1, n0 = int(t.sum()), int((~t).sum())
    if n1 == 0 or n0 == 0:
        return np.nan
    r = s.rank()
    return (r[t].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def ols_slope(x, y):
    m = x.notna() & y.notna()
    return np.polyfit(x[m].astype(float), y[m].astype(float), 1)[0]

def fmt(t, d=2):
    return {"value": round(t[0], d), "ci_low": round(t[1], d), "ci_high": round(t[2], d)}

ROB = {}
g = acc[acc["peak_later"].notna()].copy()
g["peak"] = g["peak_later"].astype(float)
g["gap_f"] = g["gap"].astype(float)
print(f"Players with a later edition: {len(g):,}")

Players with a later edition: 8,761


### 4.1 Forecast or ceiling? Calibration of potential
If potential were an honest forecast, the slope of peak later rating on potential would be about 1, and about half of
each group would beat it. A slope below 1 means the extremes are overconfident: high potentials promise too much,
low potentials too little.

In [12]:
slope = boot_ci(g, lambda d: ols_slope(d["potential"], d["peak"]))
print(f"Calibration slope: {slope[0]:.2f} (95% CI {slope[1]:.2f}–{slope[2]:.2f})")
check("Calibration slope CI excludes 1 (potential is not calibrated)", slope[2] < 1, f"{slope[0]:.2f} [{slope[1]:.2f}, {slope[2]:.2f}]")
bins = [0, 60, 65, 70, 75, 80, 85, 99]
labels = ["≤60", "61–65", "66–70", "71–75", "76–80", "81–85", "86+"]
g["pot_bin"] = pd.cut(g["potential"], bins, labels=labels)
calib = g.groupby("pot_bin", observed=True).agg(n=("peak", "size"), mean_potential=("potential", "mean"),
                                                 mean_peak=("peak", "mean"), beat_or_met=("gap_f", lambda s: (s <= 0).mean()))
display(calib.round(2))
ROB["calibration"] = {"slope": fmt(slope),
                      "points_of_peak_per_10_points_potential": round(slope[0] * 10, 1),
                      "by_potential": {k: {"n": int(r.n), "mean_potential": round(r.mean_potential, 1), "mean_peak": round(r.mean_peak, 1),
                                           "beat_or_met_pct": fd.pct(r.beat_or_met)} for k, r in calib.iterrows()}}

Calibration slope: 0.61 (95% CI 0.59–0.63)
PASS - Calibration slope CI excludes 1 (potential is not calibrated) | 0.61 [0.59, 0.63]


,n,mean_potential,mean_peak,beat_or_met
pot_bin,,,,
≤60,526,57.86,60.15,0.60
61–65,1403,63.47,63.11,0.50
66–70,2550,68.11,65.13,0.33
71–75,2420,72.99,68.18,0.23
76–80,1361,77.63,71.29,0.17
81–85,445,82.43,75.95,0.16
86+,56,86.77,80.55,0.14


### 4.2 Did EA inflate the promise? (whole database, every edition; population counts, no sampling)

In [13]:
pan = pd.read_parquet(config.CLEAN / "fifa_panel.parquet", columns=["player_id", "edition", "edition_label", "age", "overall", "potential"])
u = pan[pan["age"] <= config.MAX_ENTRY_AGE]
infl = u.groupby(["edition", "edition_label"]).agg(u21=("player_id", "size"), mean_potential=("potential", "mean"),
                                                   mean_overall=("overall", "mean"),
                                                   wonderkids=("potential", lambda s: int((s >= config.WONDERKID_POTENTIAL).sum())))
infl["headroom"] = infl["mean_potential"] - infl["mean_overall"]
infl["wonderkids_per_1000_u21"] = infl["wonderkids"] / infl["u21"] * 1000
top300 = pan.sort_values("overall", ascending=False).groupby("edition").head(300).groupby("edition")["overall"].mean()
infl["top300_mean_overall"] = infl.index.get_level_values("edition").map(top300)
infl = infl.reset_index("edition")
display(infl.round(2))
peak_ed = infl["wonderkids_per_1000_u21"].idxmax()
ROB["inflation"] = {
    "by_edition": {k: {"u21": int(r.u21), "mean_potential": round(r.mean_potential, 1), "mean_overall": round(r.mean_overall, 1),
                       "headroom": round(r.headroom, 1), "wonderkids_per_1000": round(r.wonderkids_per_1000_u21, 1),
                       "top300_mean_overall": round(r.top300_mean_overall, 1)} for k, r in infl.iterrows()},
    "headroom_first": round(infl["headroom"].iat[0], 1), "headroom_max": round(infl["headroom"].max(), 1),
    "headroom_max_edition": infl["headroom"].idxmax(), "headroom_last": round(infl["headroom"].iat[-1], 1),
    "wk_per_1000_first": round(infl["wonderkids_per_1000_u21"].iat[0], 1),
    "wk_per_1000_max": round(infl["wonderkids_per_1000_u21"].max(), 1), "wk_per_1000_max_edition": peak_ed,
    "wk_per_1000_last": round(infl["wonderkids_per_1000_u21"].iat[-1], 1),
    "mean_overall_change_first_to_max": round(infl.loc[peak_ed, "mean_overall"] - infl["mean_overall"].iat[0], 1),
    "mean_potential_change_first_to_max": round(infl.loc[peak_ed, "mean_potential"] - infl["mean_potential"].iat[0], 1),
    "top300_drift_first_to_max": round(infl.loc[peak_ed, "top300_mean_overall"] - infl["top300_mean_overall"].iat[0], 1),
}
print(json_like := {k: v for k, v in ROB["inflation"].items() if k != "by_edition"})

,edition,u21,mean_potential,mean_overall,wonderkids,headroom,wonderkids_per_1000_u21,top300_mean_overall
edition_label,,,,,,,,
FIFA 15,15,4594,68.38,58.82,48,9.55,10.45,82.13
FIFA 16,16,4663,70.40,59.97,82,10.43,17.59,82.66
FIFA 17,17,4578,72.04,60.45,106,11.58,23.15,83.83
FIFA 18,18,4522,72.80,60.59,137,12.20,30.30,84.02
FIFA 19,19,4542,72.99,60.45,144,12.54,31.70,84.39
FIFA 20,20,4564,73.61,60.57,170,13.04,37.25,84.39
FIFA 21,21,4799,73.05,60.02,145,13.04,30.21,83.99
FIFA 22,22,5047,73.12,60.52,120,12.60,23.78,84.03
FIFA 23,23,4642,73.05,60.52,127,12.53,27.36,83.95


{'headroom_first': np.float64(9.6), 'headroom_max': np.float64(13.0), 'headroom_max_edition': 'FIFA 20', 'headroom_last': np.float64(12.7), 'wk_per_1000_first': np.float64(10.4), 'wk_per_1000_max': np.float64(37.2), 'wk_per_1000_max_edition': 'FIFA 20', 'wk_per_1000_last': np.float64(23.5), 'mean_overall_change_first_to_max': np.float64(1.7), 'mean_potential_change_first_to_max': np.float64(5.2), 'top300_drift_first_to_max': np.float64(2.3)}


### 4.3 Does potential beat simple benchmarks and the market?
- **Benchmark:** a linear prediction of peak later rating from overall and age only, fitted out-of-sample (trained on the other
  three entry editions, applied to the held-out one).
- **Market:** Transfermarkt market value at entry (latest valuation on or before the entry date).
- Ranking quality is measured as Spearman with peak later rating, and AUC for real-life "made it" (matched entrants).

In [14]:
def loeo_benchmark(df):
    # Leave-one-entry-edition-out OLS: peak ~ 1 + overall + age, fitted on non-vanished entrants of the other editions
    pred = pd.Series(np.nan, index=df.index)
    for e in df["entry_edition"].unique():
        tr = df[(df["entry_edition"] != e) & df["peak_later"].notna()]
        X = np.column_stack([np.ones(len(tr)), tr["overall"], tr["age"]])
        coef, *_ = np.linalg.lstsq(X, tr["peak_later"].astype(float), rcond=None)
        te = df["entry_edition"] == e
        pred[te] = coef[0] + coef[1] * df.loc[te, "overall"] + coef[2] * df.loc[te, "age"]
    return pred

acc_b = acc.copy()
acc_b["benchmark"] = loeo_benchmark(acc_b)
acc_b["log_value_entry"] = np.log1p(acc_b["market_value_at_entry_eur"])
g2 = acc_b[acc_b["peak_later"].notna()].copy(); g2["peak"] = g2["peak_later"].astype(float)

rows = []
for name, col in [("Potential", "potential"), ("Overall (current rating)", "overall"), ("Benchmark: overall + age", "benchmark")]:
    rows.append({"predictor": name, "target": "peak later rating (Spearman)", **fmt(boot_ci(g2, lambda d, c=col: spearman(d[c], d["peak"])), 3)})
diff_sp = boot_ci(g2, lambda d: spearman(d["potential"], d["peak"]) - spearman(d["benchmark"], d["peak"]))

m = acc_b[acc_b["is_matched"]]
mv = m[m["log_value_entry"].notna()]
for name, col in [("Potential", "potential"), ("Benchmark: overall + age", "benchmark"), ("Market value at entry", "log_value_entry")]:
    rows.append({"predictor": name, "target": "made it, matched with a value at entry (AUC)",
                 **fmt(boot_ci(mv, lambda d, c=col: auc(d[c], d["made_it"])), 3)})
diff_auc_bench = boot_ci(mv, lambda d: auc(d["potential"], d["made_it"]) - auc(d["benchmark"], d["made_it"]))
diff_auc_market = boot_ci(mv, lambda d: auc(d["potential"], d["made_it"]) - auc(d["log_value_entry"], d["made_it"]))
bench_tbl = pd.DataFrame(rows)
display(bench_tbl)
print(f"Potential − benchmark (Spearman with peak): {diff_sp[0]:+.3f} [{diff_sp[1]:+.3f}, {diff_sp[2]:+.3f}]")
print(f"Potential − benchmark (AUC made it):        {diff_auc_bench[0]:+.3f} [{diff_auc_bench[1]:+.3f}, {diff_auc_bench[2]:+.3f}]")
print(f"Potential − market value (AUC made it):     {diff_auc_market[0]:+.3f} [{diff_auc_market[1]:+.3f}, {diff_auc_market[2]:+.3f}]")

# Does potential add information beyond market value? AUC of potential within market-value quintiles
mv = mv.assign(value_quintile=pd.qcut(mv["log_value_entry"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]))
within = mv.groupby("value_quintile", observed=True).apply(lambda d: auc(d["potential"], d["made_it"]), include_groups=False)
within_ci = boot_ci(mv, lambda d: np.nanmean([auc(x["potential"], x["made_it"]) for _, x in d.groupby("value_quintile", observed=True)]))
print("AUC of potential within market-value quintiles:", within.round(3).to_dict(), f"mean {within_ci[0]:.3f} [{within_ci[1]:.3f}, {within_ci[2]:.3f}]")
ROB["benchmark"] = {
    "table": bench_tbl.to_dict("records"),
    "potential_minus_benchmark_spearman": fmt(diff_sp, 3),
    "potential_minus_benchmark_auc": fmt(diff_auc_bench, 3),
    "potential_minus_market_auc": fmt(diff_auc_market, 3),
    "potential_auc_within_value_quintiles_mean": fmt(within_ci, 3),
    "n_matched_with_value": len(mv), "share_matched_with_value_pct": fd.pct(len(mv) / len(m)),
}

,predictor,target,value,ci_low,ci_high
0,Potential,peak later rating (Spearman),0.497,0.480,0.513
1,Overall (current rating),peak later rating (Spearman),0.569,0.551,0.583
2,Benchmark: overall + age,peak later rating (Spearman),0.560,0.545,0.576
3,Potential,"made it, matched with a value at entry (AUC)",0.713,0.696,0.732
4,Benchmark: overall + age,"made it, matched with a value at entry (AUC)",0.700,0.679,0.719
5,Market value at entry,"made it, matched with a value at entry (AUC)",0.716,0.698,0.733


Potential − benchmark (Spearman with peak): -0.063 [-0.074, -0.052]
Potential − benchmark (AUC made it):        +0.013 [+0.001, +0.024]
Potential − market value (AUC made it):     -0.002 [-0.020, +0.016]


AUC of potential within market-value quintiles: {1: 0.59, 2: 0.577, 3: 0.62, 4: 0.637, 5: 0.661} mean 0.617 [0.592, 0.641]


### 4.4 Overconfidence by age at entry
Does the extra headroom EA gives younger players show up in their careers?

In [15]:
age_tbl = []
for a_, d in g.groupby("age"):
    gp = boot_ci(d, lambda x: x["gap_f"].mean())
    mi = boot_ci(d, lambda x: x["made_it"].mean())
    age_tbl.append({"age_at_entry": int(a_), "n": len(d), "headroom": round((d["potential"] - d["overall"]).mean(), 1),
                    "mean_gap": round(gp[0], 2), "gap_ci": f"{gp[1]:.2f}–{gp[2]:.2f}",
                    "made_it_pct": round(mi[0] * 100, 1), "made_it_ci": f"{mi[1]*100:.1f}–{mi[2]*100:.1f}"})
age_tbl = pd.DataFrame(age_tbl)
display(age_tbl)
young, old = g[g["age"] <= 17], g[g["age"] == 21]
gap_diff = boot_ci(pd.concat([young.assign(grp=1), old.assign(grp=0)]),
                   lambda d: d.loc[d["grp"] == 1, "gap_f"].mean() - d.loc[d["grp"] == 0, "gap_f"].mean())
mi_diff = boot_ci(pd.concat([young.assign(grp=1), old.assign(grp=0)]),
                  lambda d: d.loc[d["grp"] == 1, "made_it"].mean() - d.loc[d["grp"] == 0, "made_it"].mean())
print(f"Gap, age ≤17 minus age 21: {gap_diff[0]:+.2f} [{gap_diff[1]:+.2f}, {gap_diff[2]:+.2f}]")
print(f"Made it (share), age ≤17 minus age 21: {mi_diff[0]*100:+.1f} pts [{mi_diff[1]*100:+.1f}, {mi_diff[2]*100:+.1f}]")
check("Age effect: gap CI (≤17 vs 21) excludes 0", gap_diff[1] > 0, f"{gap_diff[0]:+.2f}")
ROB["age"] = {"by_age": age_tbl.to_dict("records"), "gap_17_minus_21": fmt(gap_diff),
              "made_it_17_minus_21_pts": {k: round(v * 100, 1) for k, v in fmt(mi_diff, 4).items()}}

,age_at_entry,n,headroom,mean_gap,gap_ci,made_it_pct,made_it_ci
0,16,72,14.9,5.42,3.46–7.35,16.7,8.3–25.0
1,17,862,15.4,5.47,4.94–5.97,14.8,12.5–17.3
2,18,1790,13.7,4.87,4.55–5.19,15.3,13.6–16.9
3,19,2088,11.9,4.10,3.80–4.38,15.5,13.8–17.0
4,20,2041,9.8,2.46,2.19–2.72,13.8,12.4–15.2
5,21,1908,8.2,1.58,1.31–1.84,13.7,12.2–15.4


Gap, age ≤17 minus age 21: +3.88 [+3.31, +4.50]
Made it (share), age ≤17 minus age 21: +1.3 pts [-1.3, +4.0]
PASS - Age effect: gap CI (≤17 vs 21) excludes 0 | +3.88


### 4.5 Club strength among elite prospects (potential ≥ 80)
Across the whole cohort the club-Elo link is near zero; within elite prospects it is tested with quartiles and with a regression that
holds potential and age constant.

In [16]:
hi = g[(g["potential"] >= 80) & g["club_elo_at_entry"].notna()].copy()
hi["elo_q"] = pd.qcut(hi["club_elo_at_entry"], 4, labels=["Q1 weakest", "Q2", "Q3", "Q4 strongest"])
q_tbl = []
for q, d in hi.groupby("elo_q", observed=True):
    gp, mi = boot_ci(d, lambda x: x["gap_f"].mean()), boot_ci(d, lambda x: x["made_it"].mean())
    q_tbl.append({"club_elo_quartile": q, "n": len(d), "elo_range": f"{d['club_elo_at_entry'].min():.0f}–{d['club_elo_at_entry'].max():.0f}",
                  "mean_potential": round(d["potential"].mean(), 1), "mean_gap": round(gp[0], 2), "gap_ci": f"{gp[1]:.2f}–{gp[2]:.2f}",
                  "made_it_pct": round(mi[0] * 100, 1), "made_it_ci": f"{mi[1]*100:.1f}–{mi[2]*100:.1f}"})
q_tbl = pd.DataFrame(q_tbl); display(q_tbl)

def elo_coef(d):
    # gap ~ 1 + Elo/100 + potential + age: Elo coefficient (points of gap per 100 Elo)
    X = np.column_stack([np.ones(len(d)), d["club_elo_at_entry"] / 100, d["potential"], d["age"]])
    return np.linalg.lstsq(X, d["gap_f"], rcond=None)[0][1]
coef = boot_ci(hi, elo_coef)
rho_hi = boot_ci(hi, lambda d: spearman(d["club_elo_at_entry"], d["gap_f"]))
rho_all = boot_ci(g[g["club_elo_at_entry"].notna()], lambda d: spearman(d["club_elo_at_entry"], d["gap_f"]))
print(f"Potential 80+: gap change per +100 club Elo, holding potential and age: {coef[0]:+.2f} [{coef[1]:+.2f}, {coef[2]:+.2f}]")
print(f"Spearman(club Elo, gap): potential 80+ {rho_hi[0]:.2f} [{rho_hi[1]:.2f}, {rho_hi[2]:.2f}]; all {rho_all[0]:.2f} [{rho_all[1]:.2f}, {rho_all[2]:.2f}]")
check("Elite-club effect: Elo coefficient CI excludes 0", coef[2] < 0 or coef[1] > 0, f"{coef[0]:+.2f}")
ROB["club_effect"] = {"potential_80plus_quartiles": q_tbl.to_dict("records"), "gap_per_100_elo_controlled": fmt(coef),
                      "spearman_elo_gap_80plus": fmt(rho_hi), "spearman_elo_gap_all": fmt(rho_all), "n_80plus_with_elo": len(hi)}

,club_elo_quartile,n,elo_range,mean_potential,mean_gap,gap_ci,made_it_pct,made_it_ci
0,Q1 weakest,148,1172–1570,81.7,8.01,7.05–8.81,38.5,30.4–46.0
1,Q2,148,1570–1662,82.0,6.12,5.16–7.05,54.1,45.9–62.2
2,Q3,148,1666–1788,82.2,5.46,4.60–6.39,58.1,50.0–66.2
3,Q4 strongest,147,1789–2082,82.9,4.90,3.92–5.90,64.6,57.1–72.1


Potential 80+: gap change per +100 club Elo, holding potential and age: -0.78 [-1.09, -0.47]
Spearman(club Elo, gap): potential 80+ -0.19 [-0.27, -0.11]; all 0.02 [-0.00, 0.05]
PASS - Elite-club effect: Elo coefficient CI excludes 0 | -0.78


### 4.6 Nations: which mean gaps are distinguishable from the cohort average?

In [17]:
cohort_mean = g["gap_f"].mean()
nat_rows = []
for n_, d in g.groupby("nationality_name"):
    if len(d) < config.CHART_MIN_NATION_N:
        continue
    t = boot_ci(d, lambda x: x["gap_f"].mean())
    nat_rows.append({"nation": n_, "n": len(d), "mean_gap": round(t[0], 2), "ci_low": round(t[1], 2), "ci_high": round(t[2], 2),
                     "vs_cohort": "above" if t[1] > cohort_mean else ("below" if t[2] < cohort_mean else "not distinguishable")})
nat_ci = pd.DataFrame(nat_rows).sort_values("mean_gap")
display(nat_ci[nat_ci["vs_cohort"] != "not distinguishable"])
print(f"Cohort mean gap {cohort_mean:.2f}. Nations (n ≥ {config.CHART_MIN_NATION_N}): {len(nat_ci)}; "
      f"distinguishable: {int((nat_ci['vs_cohort'] != 'not distinguishable').sum())}")
eng_vs = g[g["nationality_name"].isin(["England", "Spain", "France"])].assign(grp=lambda d: d["nationality_name"].eq("England"))
eng_diff = boot_ci(eng_vs, lambda d: d.loc[d["grp"], "gap_f"].mean() - d.loc[~d["grp"], "gap_f"].mean())
print(f"England minus Spain+France: {eng_diff[0]:+.2f} [{eng_diff[1]:+.2f}, {eng_diff[2]:+.2f}]")
ROB["nations"] = {"cohort_mean_gap": round(cohort_mean, 2), "table": nat_ci.to_dict("records"),
                  "above_average": nat_ci.loc[nat_ci["vs_cohort"] == "above", "nation"].tolist(),
                  "below_average": nat_ci.loc[nat_ci["vs_cohort"] == "below", "nation"].tolist(),
                  "england_minus_spain_france": fmt(eng_diff)}

,nation,n,mean_gap,ci_low,ci_high,vs_cohort
5,Bosnia and Herzegovina,23,0.83,-1.35,3.09,below
14,Denmark,196,0.98,0.04,2.06,below
13,Côte d'Ivoire,50,1.14,-0.64,3.14,below
3,Austria,152,1.53,0.57,2.53,below
41,Saudi Arabia,130,2.02,0.81,3.06,below
1,Argentina,435,2.06,1.41,2.75,below
10,Colombia,280,2.07,1.27,2.83,below
47,Spain,500,2.27,1.66,2.86,below
17,France,604,2.29,1.67,2.84,below
36,Poland,190,2.36,1.32,3.37,below


Cohort mean gap 3.47. Nations (n ≥ 15): 56; distinguishable: 20


England minus Spain+France: +1.76 [+1.20, +2.36]


### 4.7 Alternative success measures
- **Big-5, 3,000+ min** (default); **within 5 years** of entry (equal follow-up for every cohort); **any covered top flight**
  (Transfermarkt's first-tier leagues, so a Benfica or Ajax career counts); **peak market value ≥ €10m** after entry.
All shares are of all entrants (unmatched = no).

In [18]:
for c_ in [f"made_it_{config.FOLLOWUP_YEARS}y", "made_it_top_flight"]:
    acc[c_] = acc[c_].fillna(False).astype(bool)
acc["value_star"] = (acc["peak_market_value_eur"] >= config.PEAK_VALUE_STAR_EUR).fillna(False)
meas = {"Big-5 3,000+ min": "made_it", f"Big-5 within {config.FOLLOWUP_YEARS} years": f"made_it_{config.FOLLOWUP_YEARS}y",
        "Any covered top flight 3,000+ min": "made_it_top_flight", "Peak value ≥ €10m": "value_star"}
alt = acc.groupby("potential_band", observed=True)[list(meas.values())].mean().mul(100).round(1)
alt.columns = list(meas.keys()); alt.index = alt.index.astype(str)
alt_ed = acc.groupby("entry_edition_label")[list(meas.values())].mean().mul(100).round(1); alt_ed.columns = list(meas.keys())
display(alt); display(alt_ed)
mono = all(alt[c].is_monotonic_increasing for c in alt.columns)
check("Every success measure rises with the potential band", mono)
auc_alt = {k: round(auc(acc["potential"], acc[c]), 3) for k, c in meas.items()}
print("AUC of potential for each measure (all entrants):", auc_alt)
ROB["alt_success"] = {"by_band_pct": {b: {k: int(round(v)) for k, v in r.items()} for b, r in alt.iterrows()},
                      "by_entry_edition_pct": {e: {k: int(round(v)) for k, v in r.items()} for e, r in alt_ed.iterrows()},
                      "potential_auc": auc_alt, "rises_with_band_for_every_measure": mono}

,"Big-5 3,000+ min",Big-5 within 5 years,"Any covered top flight 3,000+ min",Peak value ≥ €10m
potential_band,,,,
<70,3.9,1.8,9.5,2.5
70–74,9.7,5.9,21.7,6.1
75–79,22.4,14.5,39.8,16.5
80–84,45.8,35.5,63.7,43.3
85+,68.3,62.5,80.8,75.0


,"Big-5 3,000+ min",Big-5 within 5 years,"Any covered top flight 3,000+ min",Peak value ≥ €10m
entry_edition_label,,,,
FIFA 15,14.1,9.7,25.3,9.9
FIFA 16,9.0,5.1,17.9,6.6
FIFA 17,9.6,6.2,18.9,8.0
FIFA 18,8.9,5.4,17.8,8.0


PASS - Every success measure rises with the potential band 
AUC of potential for each measure (all entrants): {'Big-5 3,000+ min': np.float64(0.777), 'Big-5 within 5 years': np.float64(0.809), 'Any covered top flight 3,000+ min': np.float64(0.751), 'Peak value ≥ €10m': np.float64(0.81)}


### 4.8 Why do players vanish?
"Vanished" means no later FIFA edition. That can be a failed career, or the game losing the league's licence, or a move to an
unlicensed league.

In [19]:
van = acc[acc["status"] == V].copy()
pan_l = pd.read_parquet(config.CLEAN / "fifa_panel.parquet", columns=["edition", "league_id"])
leagues_by_ed = pan_l.dropna().groupby("edition")["league_id"].apply(set).to_dict()
van["league_dropped_next_edition"] = [lid not in leagues_by_ed.get(e + 1, set()) if pd.notna(lid) else False
                                      for lid, e in zip(van["league_id"], van["entry_edition"])]
van["kept_playing_top_flight"] = van["league_minutes_all_covered"].fillna(0) > 0
van_tbl = pd.Series({
    "vanished": len(van),
    "matched to Transfermarkt %": round(van["is_matched"].mean() * 100, 1),
    "entry league left the game next edition %": round(van["league_dropped_next_edition"].mean() * 100, 1),
    "kept playing in a covered top flight %": round(van["kept_playing_top_flight"].mean() * 100, 1),
    "made it (Big-5 3,000+) %": round(van["made_it"].mean() * 100, 1),
    "potential 75+": int((van["potential"] >= 75).sum()),
})
display(van_tbl.to_frame("value"))
print("Entry leagues of vanished players whose league left the game:",
      van[van["league_dropped_next_edition"]]["league_name_edition"].value_counts().head(8).to_dict())
ROB["vanished"] = {k: (float(v) if isinstance(v, float) else int(v)) for k, v in van_tbl.items()}

,value
vanished,2758.0
matched to Transfermarkt %,28.4
entry league left the game next edition %,0.1
kept playing in a covered top flight %,8.8
"made it (Big-5 3,000+) %",0.1
potential 75+,167.0


Entry leagues of vanished players whose league left the game: {'Scottish Championship': 2}


### 4.9 Self-review: threats to the findings
Four ways the results above could be artifacts of the method, each tested:
1. **Survivorship:** "peak later" exists only for players who stayed in the game, and low-potential players who didn't improve
   tend to be dropped, which could flatten the calibration slope on its own.
2. **Mechanical group effects:** with a slope below 1, any group with higher average potential shows a larger raw gap.
   Group comparisons therefore use **shortfall vs similar prospects** = typical peak for that potential (calibration line) −
   actual peak (positive = fell further short than peers with the same potential).
3. **Club bias in the outcome:** EA's *later* ratings may themselves favour big clubs, so club effects are re-checked with a
   club-neutral real outcome (3,000+ minutes in any covered top flight).
4. **League mix:** England's gap may come from English League One/Two players in the cohort.

In [20]:
from src import stats as st
line = st.calibration_line(g)
g["shortfall"] = st.shortfall(g, line)
REV = {"calibration_line": {"intercept": round(line[0], 3), "slope": round(line[1], 3)}}

# 1. Survivorship
pan_p = pd.read_parquet(config.CLEAN / "fifa_panel.parquet", columns=["player_id", "edition"])
pres = pan_p.merge(acc[["player_id", "entry_edition"]], on="player_id")
n_next4 = pres[(pres["edition"] > pres["entry_edition"]) & (pres["edition"] <= pres["entry_edition"] + 4)].groupby("player_id").size()
full4 = g[g["player_id"].map(n_next4).fillna(0) == 4]
imp = acc.assign(peak=acc["peak_later"].astype(float).fillna(acc["overall"]))
s_full = boot_ci(full4, lambda d: ols_slope(d["potential"], d["peak"]))
s_imp = boot_ci(imp, lambda d: ols_slope(d["potential"], d["peak"]))
print(f"Calibration slope: baseline {slope[0]:.2f}; only players in all 4 next editions {s_full[0]:.2f} [{s_full[1]:.2f}, {s_full[2]:.2f}] "
      f"(n={len(full4):,}); vanished given peak = entry overall {s_imp[0]:.2f} [{s_imp[1]:.2f}, {s_imp[2]:.2f}]")
check("Overconfidence survives every survivorship treatment (all slope CIs below 1)", max(slope[2], s_full[2], s_imp[2]) < 1)
REV["survivorship"] = {"slope_baseline": fmt(slope), "slope_full_survivors": fmt(s_full), "slope_vanished_as_no_growth": fmt(s_imp),
                       "n_full_survivors": len(full4)}

# Chart 7 with survivorship removed
lc_p = pan_p.merge(lc[["player_id", "entry_edition"]], on="player_id")
lc_n4 = lc_p[(lc_p["edition"] > lc_p["entry_edition"]) & (lc_p["edition"] <= lc_p["entry_edition"] + 4)].groupby("player_id").size()
lc_full = lca[lca["player_id"].map(lc_n4).fillna(0) == 4]
learn_full = lc_full.groupby("entry_edition_label").agg(n=("player_id", "size"),
                 reached_within_4=("status_within_4", lambda s: (s == R).mean()),
                 mean_gap_within_4=("gap_within_4", lambda s: s.astype(float).mean()))
display(learn_full.round(3))
check("Chart 7 decline survives survivorship control", learn_full["reached_within_4"].iat[-1] < learn_full["reached_within_4"].iat[0])
REV["chart07_full_survivors"] = {k: {"n": int(r.n), "reached_within_4_pct": fd.pct(r.reached_within_4),
                                     "mean_gap_within_4": fd.gap1(r.mean_gap_within_4)} for k, r in learn_full.iterrows()}

# 2. Group effects on shortfall vs similar prospects
def grp_ci(col, min_n=15):
    rows = []
    for k_, d in g.groupby(col, observed=True):
        if len(d) < min_n:
            continue
        t = boot_ci(d, lambda x: x["shortfall"].mean())
        rows.append({col: k_, "n": len(d), "shortfall": round(t[0], 2), "ci_low": round(t[1], 2), "ci_high": round(t[2], 2)})
    return pd.DataFrame(rows)
age_sf = grp_ci("age"); pos_sf = grp_ci("primary_position", 30).sort_values("shortfall"); grp_sf = grp_ci("position_group")
display(age_sf); display(pos_sf); display(grp_sf)
young_old = boot_ci(pd.concat([g[g["age"] <= 17].assign(k=1), g[g["age"] == 21].assign(k=0)]),
                    lambda d: d.loc[d["k"] == 1, "shortfall"].mean() - d.loc[d["k"] == 0, "shortfall"].mean())
check("Age effect survives the potential adjustment (≤17 vs 21 CI excludes 0)", young_old[1] > 0, f"{young_old[0]:+.2f}")
REV["age_shortfall"] = {"by_age": age_sf.to_dict("records"), "age17_minus_21": fmt(young_old)}
REV["position_shortfall"] = {"by_position": pos_sf.to_dict("records"), "by_group": grp_sf.to_dict("records")}

# 3. Club: badge premium at entry, and club effect with a club-neutral real outcome
e_ = acc[acc["club_elo_at_entry"].notna()].copy()
def badge(d):
    X = np.column_stack([np.ones(len(d)), d["overall"], d["age"], d["club_elo_at_entry"] / 100])
    return np.linalg.lstsq(X, d["potential"].astype(float), rcond=None)[0][3]
prem = boot_ci(e_, badge)
hi2 = g[(g["potential"] >= 80) & g["club_elo_at_entry"].notna()].copy()
hi2["made_it_top_flight"] = hi2["made_it_top_flight"].fillna(False).astype(bool)
def coef_on(d, y):
    X = np.column_stack([np.ones(len(d)), d["club_elo_at_entry"] / 100, d["potential"], d["age"]])
    return np.linalg.lstsq(X, d[y].astype(float), rcond=None)[0][1]
c_sf = boot_ci(hi2, lambda d: coef_on(d, "shortfall"))
c_tf = boot_ci(hi2, lambda d: coef_on(d, "made_it_top_flight"))
print(f"Badge premium at entry: +{prem[0]:.2f} potential per +100 Elo (same overall and age) [{prem[1]:.2f}, {prem[2]:.2f}]")
print(f"Potential 80+, per +100 Elo: shortfall {c_sf[0]:+.2f} [{c_sf[1]:+.2f}, {c_sf[2]:+.2f}] (EA's own later ratings); "
      f"any-top-flight made it {c_tf[0]*100:+.1f} pts [{c_tf[1]*100:+.1f}, {c_tf[2]*100:+.1f}] (club-neutral real outcome)")
club_real_confirms = c_tf[1] > 0
NOTES.append("Elite-club effect (4.5) appears in EA's own later ratings but NOT in a club-neutral real outcome "
             f"(any-top-flight: {c_tf[0]*100:+.1f} pts per +100 Elo, CI {c_tf[1]*100:+.1f} to {c_tf[2]*100:+.1f}). "
             "Not used as a headline; chart 10 shows the badge premium instead.")
REV["club"] = {"badge_premium_potential_per_100_elo": fmt(prem), "elite_shortfall_per_100_elo": fmt(c_sf),
               "elite_any_top_flight_pts_per_100_elo": {k: round(v * 100, 1) for k, v in fmt(c_tf, 4).items()},
               "real_outcome_confirms_club_effect": bool(club_real_confirms)}

# 4. England vs league tier
g["eng_l1l2"] = g["league_name_edition"].isin(["League One", "League Two"]) & g["league_country"].eq("England")
eng_all = boot_ci(g[g["nationality_name"] == "England"], lambda d: d["shortfall"].mean())
eng_top = boot_ci(g[(g["nationality_name"] == "England") & ~g["eng_l1l2"]], lambda d: d["shortfall"].mean())
spfr = boot_ci(g[g["nationality_name"].isin(["Spain", "France"])], lambda d: d["shortfall"].mean())
tier = g.groupby(g["league_level"].fillna(0).astype(int))["shortfall"].agg(["size", "mean"]).round(2)
print(f"Shortfall: England {eng_all[0]:+.2f}; England excl. League One/Two {eng_top[0]:+.2f} [{eng_top[1]:+.2f}, {eng_top[2]:+.2f}]; "
      f"Spain+France {spfr[0]:+.2f} [{spfr[1]:+.2f}, {spfr[2]:+.2f}]")
display(tier.rename(columns={"size": "n", "mean": "shortfall"}))
REV["england_vs_tier"] = {"england": fmt(eng_all), "england_excl_league_one_two": fmt(eng_top), "spain_france": fmt(spfr),
                          "shortfall_by_league_level": {int(k): {"n": int(r["size"]), "shortfall": float(r["mean"])} for k, r in tier.iterrows()}}
fd.update("self_review", REV, NB)

Calibration slope: baseline 0.61; only players in all 4 next editions 0.46 [0.43, 0.49] (n=3,700); vanished given peak = entry overall 0.78 [0.76, 0.80]
PASS - Overconfidence survives every survivorship treatment (all slope CIs below 1) 


,n,reached_within_4,mean_gap_within_4
entry_edition_label,,,
FIFA 15,1775,0.458,0.686
FIFA 16,657,0.409,1.635
FIFA 17,633,0.253,3.755
FIFA 18,635,0.214,4.748
FIFA 19,600,0.173,5.523
FIFA 20,619,0.160,6.045


PASS - Chart 7 decline survives survivorship control 


,age,n,shortfall,ci_low,ci_high
0,16,72,2.10,0.32,3.82
1,17,862,1.77,1.28,2.29
2,18,1790,1.32,1.00,1.64
3,19,2088,0.52,0.26,0.78
4,20,2041,-0.90,-1.15,-0.65
5,21,1908,-1.72,-1.95,-1.50


,primary_position,n,shortfall,ci_low,ci_high
2,CDM,497,-1.04,-1.55,-0.54
8,LW,237,-0.89,-1.60,-0.20
6,LB,622,-0.59,-1.07,-0.13
9,RB,586,-0.57,-1.06,-0.09
1,CB,1279,-0.47,-0.82,-0.14
7,LM,452,-0.36,-0.95,0.20
10,RM,437,-0.28,-0.85,0.27
0,CAM,715,-0.15,-0.62,0.32
4,CM,1229,0.09,-0.28,0.45
11,RW,283,0.14,-0.56,0.86


,position_group,n,shortfall,ci_low,ci_high
0,DEF,2499,-0.51,-0.75,-0.28
1,FWD,2015,0.55,0.28,0.83
2,GK,917,1.05,0.63,1.49
3,MID,3330,-0.24,-0.46,-0.03


PASS - Age effect survives the potential adjustment (≤17 vs 21 CI excludes 0) | +3.52


Badge premium at entry: +0.29 potential per +100 Elo (same overall and age) [0.24, 0.35]
Potential 80+, per +100 Elo: shortfall -0.78 [-1.11, -0.46] (EA's own later ratings); any-top-flight made it +1.7 pts [-1.0, +4.2] (club-neutral real outcome)


Shortfall: England +1.23; England excl. League One/Two +0.64 [+0.11, +1.14]; Spain+France -1.84 [-2.21, -1.46]


,n,shortfall
league_level,,
0,48,-1.57
1,6256,-0.20
2,1617,-0.27
3,504,2.07
4,336,2.09


In [21]:
for k, v in ROB.items():
    fd.update(f"robust_{k}", v, NB)
print("Robustness sections written:", [f"robust_{k}" for k in ROB])

Robustness sections written: ['robust_calibration', 'robust_inflation', 'robust_benchmark', 'robust_age', 'robust_club_effect', 'robust_nations', 'robust_alt_success', 'robust_vanished']


## 5. Featured five: candidates (for team approval; nothing is finalised)
From FIFA 15 entrants where possible:
1. a star who reached his potential; 2. the biggest bust; 3. a late bloomer (potential < 80, peak later ≥ 85);
4. a player whose latest club is outside Europe; 5. a player whose peak later rating equals his potential exactly.

In [22]:
show = ["player_id", "short_name", "long_name", "club_name", "nationality_name", "age", "overall", "potential", "peak_later",
        "gap", "big5_minutes", "peak_market_value_eur", "tm_current_club_name", "latest_club_tier", "match_method"]
f15a = a[a["entry_edition"] == 15].copy()
f15a["peak_market_value_eur_m"] = (f15a["peak_market_value_eur"] / 1e6).round(1)
show = [c if c != "peak_market_value_eur" else "peak_market_value_eur_m" for c in show]
cands = {
    "1 Star who reached his potential": f15a[f15a["is_wonderkid"] & (f15a["status"] == R)].sort_values(["peak_later", "potential"], ascending=False),
    "2 Biggest bust": f15a[f15a["is_wonderkid"] & f15a["peak_later"].notna()].sort_values(["gap", "potential"], ascending=False),
    "3 Late bloomer": f15a[(f15a["potential"] < 80) & (f15a["peak_later"] >= 85)].sort_values(["peak_later", "gap"], ascending=[False, True]),
    "4 Latest club outside Europe": f15a[(f15a["latest_club_tier"] == outcomes.TIER_OUTSIDE)].sort_values(["potential", "peak_later"], ascending=False),
    "5 Peak later = potential": f15a[(f15a["gap"] == 0)].sort_values(["potential", "peak_later"], ascending=False),
}
for k, df in cands.items():
    print(f"\n{k}: {len(df)} FIFA 15 candidates (top 6)")
    display(df[show].head(6))
F["featured_candidates"] = {k: [{"player_id": int(r.player_id), "player": r.short_name} for r in df.head(6).itertuples()]
                            for k, df in cands.items()}
F["featured_candidates"]["status"] = "candidates; final picks in featured_five"
FEATURED = {"Reached his potential": "R. Sterling", "Biggest bust": "R. Živković", "Late bloomer": "H. Kane",
            "Latest club outside Europe": "M. Depay", "Peak later = potential": "T. Werner"}
f_rows = {}
for role, name in FEATURED.items():
    r = f15a[f15a["short_name"] == name]
    check(f"Featured pick found once: {name}", len(r) == 1)
    r = r.iloc[0]
    f_rows[role] = {"player_id": int(r.player_id), "player": r.short_name, "long_name": r.long_name, "club_2014": r.club_name,
                    "potential": int(r.potential), "peak_later": int(r.peak_later), "gap": int(r.gap),
                    "big5_minutes": int(r.big5_minutes) if pd.notna(r.big5_minutes) else None,
                    "peak_market_value_eur_m": fd.eur_m(r.peak_market_value_eur), "latest_club": r.tm_current_club_name,
                    "latest_club_tier": r.latest_club_tier}
F["featured_five"] = {"picks": f_rows, "chosen_by": "principal data scientist review, 5 Nov 2026 (team may override)"}
display(pd.DataFrame(f_rows).T)


1 Star who reached his potential: 7 FIFA 15 candidates (top 6)


,player_id,short_name,long_name,club_name,nationality_name,age,overall,potential,peak_later,gap,big5_minutes,peak_market_value_eur_m,tm_current_club_name,latest_club_tier,match_method
49,192505,R. Lukaku,Romelu Lukaku Menama,Everton,Belgium,21,80,86,88,-2,28669.0,100.0,SSC Napoli,Big-5,auto
373,202652,R. Sterling,Raheem Sterling,Liverpool,England,19,80,86,88,-2,24283.0,160.0,Feyenoord Rotterdam,Other European top flight,auto
1026,207865,Marquinhos,Marcos Aoás Corrêa,Paris Saint Germain,Brazil,20,78,86,88,-2,24303.0,75.0,Paris Saint-Germain,Big-5,auto
140,199556,M. Verratti,Marco Verratti,Paris Saint Germain,Italy,21,80,86,87,-1,15959.0,75.0,Paris Saint-Germain,Other or lower league,auto
251,201399,M. Icardi,Mauro Emanuel Icardi Rivero,Inter,Argentina,21,76,86,87,-1,16942.0,100.0,Galatasaray,Other European top flight,auto
1298,209658,L. Goretzka,Leon Christoph Goretzka,Schalke 04,Germany,19,73,85,87,-2,20817.0,70.0,Bayern Munich,Big-5,auto



2 Biggest bust: 47 FIFA 15 candidates (top 6)


,player_id,short_name,long_name,club_name,nationality_name,age,overall,potential,peak_later,gap,big5_minutes,peak_market_value_eur_m,tm_current_club_name,latest_club_tier,match_method
1895,212210,R. Živković,Richairo Živković,Ajax,Netherlands,17,69,86,70,16,106.0,4.0,FC Emmen,Other or lower league,auto
3390,220048,S. Scuffet,Simone Scuffet,Udinese,Italy,18,71,87,75,12,6733.0,6.0,Pisa Sporting Club,Big-5,auto
131,199151,J. Iturbe,Juan Manuel Iturbe Arévalo,Roma,Argentina,21,77,88,77,11,1035.0,19.0,Aris Thessalonikis,Other or lower league,auto
419,203038,Lucas Piazón,Gustavo Lucas Domingues Piazón,Eintracht Frankfurt,Brazil,20,76,86,75,11,871.0,5.0,Avs Futebol,Other or lower league,auto
1187,208790,Z. Bakkali,Zakaria Bakkali,PSV,Belgium,18,71,85,74,11,1743.0,4.0,RKC Waalwijk,No current club,auto
1651,211176,R. Kishna,Ricardo Dennie Kishna,Ajax,Netherlands,19,73,85,75,10,1217.0,4.0,ADO Den Haag,Other or lower league,auto



3 Late bloomer: 14 FIFA 15 candidates (top 6)


,player_id,short_name,long_name,club_name,nationality_name,age,overall,potential,peak_later,gap,big5_minutes,peak_market_value_eur_m,tm_current_club_name,latest_club_tier,match_method
320,202126,H. Kane,Harry Kane,Tottenham Hotspur,England,20,71,78,90,-12,33891.0,150.0,Bayern Munich,Big-5,auto
1394,210257,Ederson,Ederson Santana de Moraes,Rio Ave,Brazil,20,69,77,89,-12,24516.0,70.0,Manchester City,Big-5,auto
2025,212622,J. Kimmich,Joshua Walter Kimmich,RB Leipzig,Germany,19,63,78,89,-11,25988.0,90.0,Bayern Munich,Big-5,auto
2722,215698,M. Maignan,Mike Maignan,Paris Saint Germain,France,18,56,74,87,-13,27307.0,45.0,AC Milan,Big-5,auto
4738,223848,S. Milinković-Savić,Sergej Milinković-Savić,Genk,Serbia,19,57,65,86,-21,21564.0,90.0,Al-Hilal Saudi Football Club,Outside Europe,auto
4208,222492,L. Sané,Leroy Aziz Sané,Schalke 04,Germany,18,60,74,86,-12,18577.0,100.0,Galatasaray,Other European top flight,auto


... (2 outputs left out)


5 Peak later = potential: 234 FIFA 15 candidates (top 6)


,player_id,short_name,long_name,club_name,nationality_name,age,overall,potential,peak_later,gap,big5_minutes,peak_market_value_eur_m,tm_current_club_name,latest_club_tier,match_method
1885,212188,T. Werner,Timo Werner,VfB Stuttgart,Germany,18,70,85,85,0,21328.0,80.0,RB Leipzig,Big-5,auto
680,204970,F. Thauvin,Florian Tristan Mariano Thauvin,Olympique de Marseille,France,21,77,84,84,0,21551.0,50.0,RC Lens,Big-5,auto
1989,212462,Alex Telles,Alex Nicolao Telles,Galatasaray,Brazil,21,72,84,84,0,5355.0,40.0,S. A. F. Botafogo,Outside Europe,auto
1101,208333,E. Can,Emre Can,Liverpool,Germany,20,74,83,83,0,22418.0,40.0,Borussia Dortmund,Big-5,auto
2066,212814,João Mário,João Mário Naval da Costa Eduardo,Sporting CP,Portugal,21,73,83,83,0,4658.0,35.0,Athlitiki Enosi Konstantinoupoleos,Other European top flight,auto
2162,213331,J. Tah,Jonathan Tah,Fortuna Düsseldorf,Germany,18,68,83,83,0,26865.0,40.0,Bayern Munich,Big-5,auto


PASS - Featured pick found once: R. Sterling 
PASS - Featured pick found once: R. Živković 
PASS - Featured pick found once: H. Kane 
PASS - Featured pick found once: M. Depay 
PASS - Featured pick found once: T. Werner 


,player_id,player,long_name,club_2014,potential,peak_later,gap,big5_minutes,peak_market_value_eur_m,latest_club,latest_club_tier
Reached his potential,202652,R. Sterling,Raheem Sterling,Liverpool,86,88,-2,24283,160.0,Feyenoord Rotterdam,Other European top flight
Biggest bust,212210,R. Živković,Richairo Živković,Ajax,86,70,16,106,4.0,FC Emmen,Other or lower league
Late bloomer,202126,H. Kane,Harry Kane,Tottenham Hotspur,78,90,-12,33891,150.0,Bayern Munich,Big-5
Latest club outside Europe,202556,M. Depay,Memphis Depay,PSV,88,85,3,15013,55.0,Sport Club Corinthians Paulista,Outside Europe
Peak later = potential,212188,T. Werner,Timo Werner,VfB Stuttgart,85,85,0,21328,80.0,RB Leipzig,Big-5


## 6. Write `outputs/findings.json` and the data-quality section

In [23]:
provisional = {"match_review_pending_players": F["match"]["pending_review_players"],
               "club_overrides_seeded_unreviewed": True,
               "note": "Re-run after the Transfermarkt match review and the club-override check."}
for k, v in F.items():
    fd.update(k, v, NB)
fd.update("provisional", provisional, NB)
print("findings.json sections:", sorted(fd.load().keys()))

checks = pd.DataFrame(CHECKS)
body = f'''
- analysis.parquet: {len(a):,} rows (one per cohort entrant), {a.shape[1]} columns.
- Already-peaked entrants excluded from accuracy metrics: {n_peaked}.
- Real-life "made it" counts unmatched entrants as not made it ({int((~a["is_matched"]).sum()):,} unmatched).
- Provisional: {F["match"]["pending_review_players"]} players still pending Transfermarkt match review.

### Calibration

{dq.df_to_md(cal.round(1).reset_index().rename(columns={"index": "Band", "potential_band": "Band"}))}

### Checks

{dq.df_to_md(checks)}
'''
dq.write_section("05", "05 Analysis master", body)

findings.json sections: ['_meta', 'calibration_by_band', 'chart_01', 'chart_02', 'chart_04', 'chart_05', 'chart_06', 'chart_07', 'chart_08', 'chart_09', 'chart_10', 'chart_11', 'chart_12', 'chart_14', 'cohort', 'featured_candidates', 'featured_five', 'match', 'provisional', 'robust_age', 'robust_alt_success', 'robust_benchmark', 'robust_calibration', 'robust_club_effect', 'robust_inflation', 'robust_nations', 'robust_vanished', 'self_review']


## 7. Summary

In [24]:
display(checks)
assert (checks["Result"] == "PASS").all(), checks[checks["Result"] == "FAIL"]

,Check,Result,Detail
0,analysis: one row per cohort entrant,PASS,"11,519"
1,Status counts sum to the cohort size,PASS,
2,Chart 2 counts match CLAUDE.md §7,PASS,
3,Chart 12: attribute list has no duplicates,PASS,37 attributes
4,Calibration slope CI excludes 1 (potential is ...,PASS,"0.61 [0.59, 0.63]"
5,Age effect: gap CI (≤17 vs 21) excludes 0,PASS,+3.88
6,Elite-club effect: Elo coefficient CI excludes 0,PASS,-0.78
7,Every success measure rises with the potential...,PASS,
8,Overconfidence survives every survivorship tre...,PASS,
9,Chart 7 decline survives survivorship control,PASS,
